# Etapa 2 - Importação e Validação dos Dados

## 2.1 - Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

print("Bibliotecas importadas com sucesso.")

## 2.2 — Leitura do CSV exportado do SQLite

In [ ]:
# Ajuste o caminho conforme a localização do seu arquivo CSV.
CAMINHO_CSV = 'C:\\Users\\bruno\\repos\\mba_xperiun_repos\\mba_xperiun_dc4\\Dados\\Tabela_Modelagem_Claude_Opus_46.csv'

df = pd.read_csv(
    CAMINHO_CSV
    , sep=';'            # ajuste se necessário (';' para CSVs brasileiros)
    , encoding='utf-8'   # ajuste se necessário ('latin-1' para acentos)
    , low_memory=False
)

print(f"Shape: {df.shape}")
print(f"Registros: {df.shape[0]:,}")
print(f"Colunas:   {df.shape[1]}")

## 2.3 - Validação Inicial

### 2.3.1 - Tipos de Dados

In [ ]:
print("=" * 76)
print("TIPOS DE DADOS")
print("=" * 76)
print(df.dtypes.to_string())

### 2.3.2 - Valores Nulos

In [ ]:
print("=" * 76)
print("VALORES NULOS POR COLUNA")
print("=" * 76)
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({
    'nulos': nulos
    , 'pct': nulos_pct
})
print(resumo_nulos[resumo_nulos['nulos'] > 0].to_string())
if resumo_nulos['nulos'].sum() == 0:
    print("Nenhum valor nulo encontrado.")

### 2.3.3 - Taxa de Churn

In [ ]:
taxa_churn = df['indicador_churn'].mean()
n_churn = df['indicador_churn'].sum()
n_nao_churn = len(df) - n_churn
print("=" * 76)
print("DISTRIBUIÇÃO DA VARIÁVEL RESPOSTA")
print("=" * 76)
print(f"Total de registros: {len(df):,}")
print(f"Churn = 1:          {n_churn:,} ({taxa_churn:.2%})")
print(f"Churn = 0:          {n_nao_churn:,} ({1 - taxa_churn:.2%})")

# Validação: taxa deve estar próxima de 16%
assert 0.10 < taxa_churn < 0.25, \
    f"ALERTA: Taxa de churn ({taxa_churn:.2%}) fora do esperado (~16%)"
print(f"\n✅ Taxa de churn ({taxa_churn:.2%}) dentro do intervalo esperado.")

### 2.3.4 - Estatísticas Descritivas das Variáveis Contínuas

In [ ]:
print("=" * 76)
print("ESTATÍSTICAS DESCRITIVAS — VARIÁVEIS CONTÍNUAS")
print("=" * 76)
continuas = [
    'tempo_cliente_meses', 'premio_medio_mensal', 'franquia_media'
    , 'comissao_percentual', 'vigencia_meses', 'premio_mensal'
    , 'comissao', 'receita_esperada', 'ratio_premio_vs_medio'
    , 'qtd_apolices_acumuladas', 'qtd_ramos_distintos'
    , 'qtd_coberturas_distintas'
]
# Filtra apenas colunas que existam no DataFrame
continuas_existentes = [c for c in continuas if c in df.columns]
print(df[continuas_existentes].describe().round(2).to_string())

### 2.3.5 - Distribuição das Variáveis Categóricas

In [ ]:
print("=" * 76)
print("DISTRIBUIÇÃO — VARIÁVEIS CATEGÓRICAS")
print("=" * 76)
categoricas = [
    'tipo_cliente', 'genero', 'faixa_etaria', 'regiao'
    , 'ramo', 'cobertura', 'nome_canal', 'tipo_canal'
    , 'forma_pagamento'
]
categoricas_existentes = [c for c in categoricas if c in df.columns]
for col in categoricas_existentes:
    print(f"\n--- {col} ---")
    contagem = df[col].value_counts()
    pct = (contagem / len(df) * 100).round(2)
    resumo = pd.DataFrame({'n': contagem, '%': pct})
    print(resumo.to_string())

## 2.4 - Definição de papéis das colunas

In [ ]:
# Colunas de identificação (não entram no modelo, mas são mantidas para rastreio)
COLS_ID = [
    'apolice_id', 'cliente_id', 'produto_id', 'canal_id', 'data_inicio'
]

# Variável resposta
COL_TARGET = 'indicador_churn'

# Variáveis usadas apenas para estratificação (não entram no modelo)
COLS_ESTRATIFICACAO = ['faixa_tempo_casa']

# Variáveis binárias que já são numéricas (não precisam de encoding)
COLS_BINARIAS = [
    'is_fim_semana', 'is_feriado'
    , 'tem_auto', 'tem_vida', 'tem_residencial', 'tem_cross_sell'
]

# Variáveis contínuas
COLS_CONTINUAS = [
    'tempo_cliente_meses', 'premio_medio_mensal', 'franquia_media'
    , 'comissao_percentual', 'vigencia_meses', 'premio_mensal'
    , 'comissao', 'receita_esperada', 'ratio_premio_vs_medio'
    , 'qtd_apolices_acumuladas', 'qtd_ramos_distintos'
    , 'qtd_coberturas_distintas'
]

# Variáveis categóricas (precisarão de encoding)
COLS_CATEGORICAS = [
    'tipo_cliente', 'genero', 'faixa_etaria', 'regiao'
    , 'ramo', 'cobertura', 'nome_canal', 'tipo_canal'
    , 'forma_pagamento'
]

# Variáveis temporais que serão tratadas como categóricas
COLS_TEMPORAIS_CAT = ['mes_inicio', 'trimestre_inicio']

# ano_inicio: pode ser categórica ou contínua, vamos avaliar
COLS_TEMPORAIS_CONT = ['ano_inicio']

# Features totais (tudo que é candidato ao modelo)
COLS_FEATURES = (
    COLS_CONTINUAS
    + COLS_BINARIAS
    + COLS_CATEGORICAS
    + COLS_TEMPORAIS_CAT
    + COLS_TEMPORAIS_CONT
)

print(f"IDs:              {len(COLS_ID)}")
print(f"Target:           1 ({COL_TARGET})")
print(f"Contínuas:        {len(COLS_CONTINUAS)}")
print(f"Binárias:         {len(COLS_BINARIAS)}")
print(f"Categóricas:      {len(COLS_CATEGORICAS)}")
print(f"Temporais (cat):  {len(COLS_TEMPORAIS_CAT)}")
print(f"Temporais (cont): {len(COLS_TEMPORAIS_CONT)}")
print(f"Estratificação:   {len(COLS_ESTRATIFICACAO)}")
print(f"Total features:   {len(COLS_FEATURES)}")